In [79]:
import pandas as pd
import nibabel as nib
import numpy as np
from pydicom.uid import generate_uid
from highdicom.seg import Segmentation, SegmentationTypeValues, SegmentDescription
from highdicom.sr.coding import CodedConcept

def load_nii(nii_path):
    file = nib.load(nii_path)
    data = file.get_fdata(caching='unchanged')
    return data

sax_df = pd.read_pickle("sax_df.pkl")

In [80]:
masks = load_nii('/workspaces/Roundel-Clinical/roundel/results/masks/1.3.6.1.4.1.53684.1.1.3.1671342546.2056.1623765873.123561.nii.gz')

In [ ]:

def save_mask_as_dicom_series(masks):
    sax_df = st.session_state['sax_df']
    for slice_num, uni_slice in enumerate(sax_df.slicelocation.unique()):
        slice_df = sax_df.loc[sax_df['slicelocation'] == uni_slice]
        for time_num, uni_time in enumerate(slice_df.triggertime.unique()):
            dcm = slice_df.loc[slice_df['triggertime'] == uni_time, 'dcm'].item()
            arr = masks[:, :, slice_num, time_num].astype(dcm.pixel_array.dtype)

            dcm.SeriesDescription = 'Roundel'
            dcm.PixelData = arr.tobytes()
            dcm.Rows, dcm.Columns = arr.shape
            dcm.BitsAllocated = arr.dtype.itemsize * 8
            dcm.BitsStored = dcm.BitsAllocated
            dcm.HighBit = dcm.BitsStored - 1
            dcm.RescaleSlope = 1
            dcm.RescaleIntercept = 0
            dcm.SeriesInstanceUID = dcm.SeriesInstanceUID + '000111222'
            dcm.SOPInstanceUID = dcm.SOPInstanceUID + '000111222'
            dcm.save_as(f'/workspaces/Roundel-Clinical/roundel/results/masks/dicoms/slice_{slice_num:02}_time_{time_num:02}.dcm')

/usr/local/lib/python3.8/dist-packages/pydicom/valuerep.py:443: UserWarning: The value length (24) exceeds the maximum length of 16 allowed for VR SH.
  warnings.warn(msg)


Dataset.file_meta -------------------------------
(0002, 0000) File Meta Information Group Length  UL: 202
(0002, 0001) File Meta Information Version       OB: b'\x00\x01'
(0002, 0002) Media Storage SOP Class UID         UI: MR Image Storage
(0002, 0003) Media Storage SOP Instance UID      UI: 1.3.6.1.4.1.53684.1.1.4.1671342546.2056.1623765896.125232
(0002, 0010) Transfer Syntax UID                 UI: Explicit VR Little Endian
(0002, 0012) Implementation Class UID            UI: 1.3.6.1.4.1.53684.1.0.3.6.0
(0002, 0013) Implementation Version Name         SH: 'CVI42_DCMTK_360'
-------------------------------------------------
(0008, 0005) Specific Character Set              CS: 'ISO_IR 100'
(0008, 0008) Image Type                          CS: ['ORIGINAL', 'PRIMARY', 'M', 'RETRO', 'NORM', 'DIS2D', 'FM4_2', 'FIL']
(0008, 0012) Instance Creation Date              DA: '20190601'
(0008, 0013) Instance Creation Time              TM: '130813.986000'
(0008, 0016) SOP Class UID                 

/usr/local/lib/python3.8/dist-packages/pydicom/valuerep.py:443: UserWarning: The value length (66) exceeds the maximum length of 64 allowed for VR UI. Please see <https://dicom.nema.org/medical/dicom/current/output/html/part05.html#table_6.2-1> for allowed values for each VR.
  warnings.warn(msg)


In [ ]:
# import numpy as np
# from pydicom.uid import generate_uid
# from highdicom.seg import Segmentation, SegmentDescription
# from highdicom.sr.coding import CodedConcept

# # segment descriptions for 4 labels (same as before)
# segments = [
#     SegmentDescription(
#         segment_number=1,
#         segment_label="LV",
#         segmented_property_category=CodedConcept(value="T-32000", meaning="Heart chamber", scheme_designator="SRT"),
#         segmented_property_type=CodedConcept(value="T-32000", meaning="Left ventricle", scheme_designator="SRT"),
#         algorithm_type="MANUAL"
#     ),
#     SegmentDescription(
#         segment_number=2,
#         segment_label="RV",
#         segmented_property_category=CodedConcept(value="T-32000", meaning="Heart chamber", scheme_designator="SRT"),
#         segmented_property_type=CodedConcept(value="T-32001", meaning="Right ventricle", scheme_designator="SRT"),
#         algorithm_type="MANUAL"
#     ),
#     SegmentDescription(
#         segment_number=3,
#         segment_label="LV myocardium",
#         segmented_property_category=CodedConcept(value="T-28000", meaning="Myocardium", scheme_designator="SRT"),
#         segmented_property_type=CodedConcept(value="T-28001", meaning="Left ventricular myocardium", scheme_designator="SRT"),
#         algorithm_type="MANUAL"
#     ),
#     SegmentDescription(
#         segment_number=4,
#         segment_label="RV myocardium",
#         segmented_property_category=CodedConcept(value="T-28000", meaning="Myocardium", scheme_designator="SRT"),
#         segmented_property_type=CodedConcept(value="T-28002", meaning="Right ventricular myocardium", scheme_designator="SRT"),
#         algorithm_type="MANUAL"
#     ),
# ]


# for slice_idx, slice_loc in enumerate(sax_df.slicelocation.unique()):
#     slice_df = sax_df[sax_df['slicelocation'] == slice_loc]

#     for time_idx, uni_time in enumerate(slice_df.triggertime.unique()):
#         # get the single source DICOM
#         source_dcm = slice_df[slice_df['triggertime'] == uni_time]['dcm'].iloc[0]

#         # extract the mask for this slice/time
#         # masks shape: (rows, cols, slices, times)
#         seg_array = masks[:, :, slice_idx, time_idx].astype(np.uint8)

#         # create Segmentation object for single slice
#         seg = Segmentation(
#             source_images=[source_dcm],
#             pixel_array=seg_array,
#             segmentation_type="BINARY",
#             segment_descriptions=segments,
#             series_instance_uid=source_dcm.SeriesInstanceUID + '000111222',
#             sop_instance_uid=source_dcm.SOPInstanceUID + '000111222',
#             instance_number=time_idx + 1,
#             series_number=slice_idx + 1,
#             manufacturer="Roundel",
#             manufacturer_model_name="CustomSegmentation",
#             software_versions="1.0",
#             device_serial_number="0000",
#             omit_empty_frames=True
#         )

#         # save one DICOM per slice/time
#         seg.save_as(f"roundel/results/masks/dicoms/slice_{slice_idx:02}_time_{time_idx:02}.dcm")


# import pydicom
# mask = pydicom.dcmread('/workspaces/Roundel-Clinical/roundel/results/masks/dicoms/slice_10_time_10.dcm')

Encoding an empty segmentation with "omit_empty_frames" set to True. Reverting to encoding all frames since omitting all frames is not possible.
Encoding an empty segmentation with "omit_empty_frames" set to True. Reverting to encoding all frames since omitting all frames is not possible.
Encoding an empty segmentation with "omit_empty_frames" set to True. Reverting to encoding all frames since omitting all frames is not possible.
Encoding an empty segmentation with "omit_empty_frames" set to True. Reverting to encoding all frames since omitting all frames is not possible.
Encoding an empty segmentation with "omit_empty_frames" set to True. Reverting to encoding all frames since omitting all frames is not possible.
Encoding an empty segmentation with "omit_empty_frames" set to True. Reverting to encoding all frames since omitting all frames is not possible.
Encoding an empty segmentation with "omit_empty_frames" set to True. Reverting to encoding all frames since omitting all frames is